# Cassava AI Root Cause Detective — Kaggle inference (T4 x2, 2-GPU parallel)

Inference-only notebook: pulls the LoRA adapter already trained and pushed to the Hugging Face Hub
(see `notebooks/Cassava_AIRCD_finetune.ipynb`) and runs generation split across both GPUs concurrently.
No training happens here.

**Kaggle notebook settings required before running** (right sidebar):
- Settings → Accelerator → **GPU T4 x2**
- Settings → Internet → **On** (needed for pip installs, the GitHub clone, and Hugging Face Hub access)

Speed rationale: a single Kaggle T4 has the same per-token throughput as Colab's T4 (same Turing
architecture, no bf16 tensor cores) — the win here isn't a faster GPU, it's that per-question generation
is fully independent, so splitting the question set across two GPUs and running two model copies
concurrently gets close to a 2x wall-clock speedup on top of everything already fixed in the training
notebook (batched `num_return_sequences`, bf16-merged model).

## 1. Install dependencies

In [ ]:
!pip install -q -U transformers peft accelerate huggingface_hub tqdm torchao

## 2. Imports and global seed

In [ ]:
import os
import re
import random
import concurrent.futures

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
DATA_DIR = "/kaggle/working/data"

GPU_COUNT = torch.cuda.device_count()
print(f"{GPU_COUNT} GPU(s) visible")

## 3. Hugging Face Hub auth

Uses a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets in the notebook editor) if present, otherwise
falls back to an interactive login prompt. `HF_REPO_ID` resolves from the logged-in account, matching
whatever the training notebook pushed the adapter to.

In [ ]:
from huggingface_hub import login, HfApi

try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = None

login(token=hf_token) if hf_token else login()

HF_USERNAME = HfApi().whoami()["name"]
HF_REPO_ID = f"{HF_USERNAME}/qwen25-1.5b-aircd-lora"
print(f"Loading adapter from https://huggingface.co/{HF_REPO_ID}")

## 4. Load data

Clones the same public repo the training notebook uses, for the competition CSVs.

In [ ]:
required = ["validation_questions.csv", "validation_target.csv", "test.csv", "SampleSubmission.csv"]
os.makedirs(DATA_DIR, exist_ok=True)
missing = [f for f in required if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    clone_dir = "/kaggle/working/cassava-network-anomaly-ai"
    if not os.path.exists(clone_dir):
        os.system(f"git clone --depth 1 'https://github.com/Ashuza11/cassava-network-anomaly-ai.git' {clone_dir}")
    for f in required:
        src = os.path.join(clone_dir, "data", f)
        if os.path.exists(src) and not os.path.exists(os.path.join(DATA_DIR, f)):
            os.system(f"cp {src} {DATA_DIR}/")
    missing = [f for f in required if not os.path.exists(os.path.join(DATA_DIR, f))]

if missing:
    raise RuntimeError(f"Missing data files after clone: {missing}")

val_q_df = pd.read_csv(os.path.join(DATA_DIR, "validation_questions.csv"))
val_t_df = pd.read_csv(os.path.join(DATA_DIR, "validation_target.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "SampleSubmission.csv"))
print(val_q_df.shape, val_t_df.shape, test_df.shape, sample_sub_df.shape)

## 5. Scoring utilities (boxed-answer extraction, position→label mapping, Pass@1)

Same logic as the training notebook's section 5 — needed here only for the local validation check.

In [ ]:
C_OPTION_RE = re.compile(r"^C(\d+):\s*(.+)$", re.MULTILINE)
PLAIN_OPTION_RE = re.compile(r"^\s*([A-Za-z0-9]+):\s*(.+)$", re.MULTILINE)
DRIVE_TABLE_MARKER = "User plane drive test data"
ENG_TABLE_MARKER = "Engeneering parameters data"
BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")

CANONICAL_DESCRIPTIONS = {
    "C1": "The serving cell's downtilt angle is too large, causing weak coverage at the far end.",
    "C2": "The serving cell's coverage distance exceeds 1km, resulting in over-shooting.",
    "C3": "A neighboring cell provides higher throughput.",
    "C4": "Non-colocated co-frequency neighboring cells cause severe overlapping coverage.",
    "C5": "Frequent handovers degrade performance.",
    "C6": "Neighbor cell and serving cell have the same PCI mod 30, leading to interference.",
    "C7": "Test vehicle speed exceeds 40km/h, impacting user throughput.",
    "C8": "Average scheduled RBs are below 160, affecting throughput.",
}


def parse_options(question_text: str):
    c_matches = C_OPTION_RE.findall(question_text)
    if c_matches:
        return [(f"C{n}", desc.strip()) for n, desc in c_matches]

    header_end = len(question_text)
    for marker in ("\nGiven:", "\n" + DRIVE_TABLE_MARKER, "\n" + ENG_TABLE_MARKER):
        idx = question_text.find(marker)
        if idx != -1:
            header_end = min(header_end, idx)
    header = question_text[:header_end]
    return [(n, desc.strip()) for n, desc in PLAIN_OPTION_RE.findall(header)]


def extract_boxed(text: str):
    if not isinstance(text, str):
        return None
    matches = BOXED_RE.findall(text)
    return matches[-1].strip() if matches else None


def map_position_to_label(question_text: str, canonical_descriptions=None):
    canonical_descriptions = canonical_descriptions or CANONICAL_DESCRIPTIONS
    desc_to_label = {desc.strip(): label for label, desc in canonical_descriptions.items()}
    options = parse_options(question_text)
    return {pos: desc_to_label.get(desc.strip(), pos) for pos, desc in options}


def pass_at_1(pred_df: pd.DataFrame, target_df: pd.DataFrame, question_lookup: dict,
              canonical_descriptions=None) -> float:
    merged = pred_df.merge(target_df, on="ID", suffixes=("_pred", "_true"))
    correct, total = 0, 0
    for _, row in merged.iterrows():
        base_id = row["ID"].rsplit("_", 1)[0]
        qtext = question_lookup.get(base_id)
        if qtext is None:
            continue
        pos_map = map_position_to_label(qtext, canonical_descriptions)
        boxed = extract_boxed(row["Target_pred"])
        pred_label = pos_map.get(boxed)
        total += 1
        if pred_label == row["Target_true"]:
            correct += 1
    return correct / total if total else 0.0

## 6. Load the merged bf16 model — one copy per visible GPU

Each copy is loaded unquantized in bf16 (~3GB, well within a T4's 16GB) and pinned to its own device, so
the two GPUs can generate independently with no cross-device communication needed.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def load_merged_model(device: str):
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    base = base.to(device)
    m = PeftModel.from_pretrained(base, HF_REPO_ID)
    m = m.merge_and_unload()
    m.eval()
    return m


devices = [f"cuda:{i}" for i in range(GPU_COUNT)] if GPU_COUNT > 0 else ["cpu"]
models = [load_merged_model(d) for d in devices]
print(f"Loaded {len(models)} model replica(s) on {devices}")

## 7. Generation helper (device-parameterized)

Seeding uses `torch.cuda.manual_seed` (device-scoped) rather than `torch.manual_seed`/`set_seed`
(process-global): the latter would reseed *every* visible GPU on each call, so two worker threads calling
it concurrently for different devices would race and clobber each other's RNG state. Scoping the seed to
`torch.cuda.set_device(device)` first keeps each thread's seeding isolated to its own GPU.

In [ ]:
GEN_TEMPERATURE = 0.4
GEN_TOP_P = 0.9
GEN_MAX_NEW_TOKENS = 300


def generate_answers(question_text, model, device, num_samples=4, base_seed=SEED):
    messages = [{"role": "user", "content": question_text}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    torch.cuda.set_device(device)
    torch.cuda.manual_seed(base_seed)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
            do_sample=True,
            temperature=GEN_TEMPERATURE,
            top_p=GEN_TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            num_return_sequences=num_samples,
        )
    prompt_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[prompt_len:], skip_special_tokens=True) for o in out]

## 8. Parallel prediction runner

Splits the question set into one chunk per GPU and runs each chunk on its own thread/model/device via
`ThreadPoolExecutor`. CUDA kernel launches release the GIL almost immediately (the actual matmuls run
async on-device), so two threads issuing generation calls to two different GPUs run close to fully
concurrently despite Python's GIL.

In [ ]:
from tqdm.auto import tqdm


def _run_chunk(chunk_df, model, device, num_samples, base_seed, progress):
    rows = []
    for _, r in chunk_df.iterrows():
        gens = generate_answers(
            r["question"], model, device,
            num_samples=num_samples,
            base_seed=base_seed + hash(r["ID"]) % 10_000,
        )
        for i, g in enumerate(gens, start=1):
            rows.append({"ID": f"{r['ID']}_{i}", "Target": g})
        progress.update(1)
    return rows


def run_predictions(question_df, num_samples=4, base_seed=SEED):
    chunks = np.array_split(question_df, len(models))
    progress = tqdm(total=len(question_df), desc="Generating")
    with concurrent.futures.ThreadPoolExecutor(max_workers=len(models)) as pool:
        futures = [
            pool.submit(_run_chunk, chunk, model, device, num_samples, base_seed, progress)
            for chunk, model, device in zip(chunks, models, devices)
        ]
        rows = [row for f in futures for row in f.result()]
    progress.close()
    return pd.DataFrame(rows)

## 9. Local validation

Same caveat as the training notebook: this only covers `validation_questions.csv`'s "full canonical
8-option, plain-digit" slice, so treat it as an optimistic upper bound, not the expected leaderboard
score.

In [ ]:
val_pred_df = run_predictions(val_q_df)
val_lookup = dict(zip(val_q_df["ID"], val_q_df["question"]))
score = pass_at_1(val_pred_df, val_t_df, val_lookup)
print(f"Local validation Pass@1: {score:.4f}")

## 10. Generate test.csv predictions and write submission.csv

In [ ]:
test_pred_df = run_predictions(test_df)
assert set(test_pred_df["ID"]) == set(sample_sub_df["ID"]), "ID mismatch vs SampleSubmission.csv"
test_pred_df = test_pred_df.set_index("ID").loc[sample_sub_df["ID"]].reset_index()
test_pred_df.to_csv("/kaggle/working/submission.csv", index=False)
test_pred_df.head()